# The Product Pricer Continued

A model that can estimate how much something costs, from its description.

## AT LAST - it's time for Fine Tuning!

After all this data preparation, and old school machine learning, we've finally arrived at the moment you've been waiting for. Fine-tuning a model.

In [35]:
# imports

import os
import re
import math
import json
import random
from dotenv import load_dotenv
from huggingface_hub import login
import matplotlib.pyplot as plt
import numpy as np
import pickle
from collections import Counter
from openai import OpenAI
from anthropic import Anthropic

In [36]:
# environment

load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
os.environ['ANTHROPIC_API_KEY'] = os.getenv('ANTHROPIC_API_KEY', 'your-key-if-not-using-env')
os.environ['HF_TOKEN'] = os.getenv('HF_TOKEN', 'your-key-if-not-using-env')

In [37]:
# Log in to HuggingFace

hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [38]:
# moved our Tester into a separate package
# call it with Tester.test(function_name, test_dataset)

from items import Item
from testing import Tester

In [39]:
openai = OpenAI()

In [40]:
%matplotlib inline

In [41]:
# Let's avoid curating all our data again! Load in the pickle files:

with open('train.pkl', 'rb') as file:
    train = pickle.load(file)

with open('test.pkl', 'rb') as file:
    test = pickle.load(file)

In [42]:
# OpenAI recommends fine-tuning with populations of 50-100 examples
# But as our examples are very small, I'm suggesting we go with 200 examples (and 1 epoch)

fine_tune_train = train[:200]
fine_tune_validation = train[200:250]

# Step 1

Prepare our data for fine-tuning in JSONL (JSON Lines) format and upload to OpenAI

In [43]:
# First let's work on a good prompt for a Frontier model
# Notice that I'm removing the " to the nearest dollar"
# When we train our own models, we'll need to make the problem as easy as possible, 
# but a Frontier model needs no such simplification.

def messages_for(item):
    system_message = "You estimate prices of items. Reply only with the price, no explanation"
    user_prompt = item.test_prompt().replace(" to the nearest dollar","").replace("\n\nPrice is $","")
    return [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_prompt},
        {"role": "assistant", "content": f"Price is ${item.price:.2f}"}
    ]

In [44]:
messages_for(train[0])

[{'role': 'system',
  'content': 'You estimate prices of items. Reply only with the price, no explanation'},
 {'role': 'user',
  'content': 'How much does this cost?\n\nDelphi FG0166 Fuel Pump Module\nDelphi brings 80 years of OE Heritage into each Delphi pump, ensuring quality and fitment for each Delphi part. Part is validated, tested and matched to the right vehicle application Delphi brings 80 years of OE Heritage into each Delphi assembly, ensuring quality and fitment for each Delphi part Always be sure to check and clean fuel tank to avoid unnecessary returns Rigorous OE-testing ensures the pump can withstand extreme temperatures Brand Delphi, Fit Type Vehicle Specific Fit, Dimensions LxWxH 19.7 x 7.7 x 5.1 inches, Weight 2.2 Pounds, Auto Part Position Unknown, Operation Mode Mechanical, Manufacturer Delphi, Model FUEL PUMP, Dimensions 19.7'},
 {'role': 'assistant', 'content': 'Price is $226.95'}]

In [45]:
# Convert the items into a list of json objects - a "jsonl" string
# Each row represents a message in the form:
# {"messages" : [{"role": "system", "content": "You estimate prices...


def make_jsonl(items):
    result = ""
    for item in items:
        messages = messages_for(item)
        messages_str = json.dumps(messages)
        result += '{"messages": ' + messages_str +'}\n'
    return result.strip()

In [46]:
print(make_jsonl(train[:3]))

{"messages": [{"role": "system", "content": "You estimate prices of items. Reply only with the price, no explanation"}, {"role": "user", "content": "How much does this cost?\n\nDelphi FG0166 Fuel Pump Module\nDelphi brings 80 years of OE Heritage into each Delphi pump, ensuring quality and fitment for each Delphi part. Part is validated, tested and matched to the right vehicle application Delphi brings 80 years of OE Heritage into each Delphi assembly, ensuring quality and fitment for each Delphi part Always be sure to check and clean fuel tank to avoid unnecessary returns Rigorous OE-testing ensures the pump can withstand extreme temperatures Brand Delphi, Fit Type Vehicle Specific Fit, Dimensions LxWxH 19.7 x 7.7 x 5.1 inches, Weight 2.2 Pounds, Auto Part Position Unknown, Operation Mode Mechanical, Manufacturer Delphi, Model FUEL PUMP, Dimensions 19.7"}, {"role": "assistant", "content": "Price is $226.95"}]}
{"messages": [{"role": "system", "content": "You estimate prices of items. 

In [47]:
# Convert the items into jsonl and write them to a file

def write_jsonl(items, filename):
    with open(filename, "w") as f:
        jsonl = make_jsonl(items)
        f.write(jsonl)

In [48]:
write_jsonl(fine_tune_train, "fine_tune_train.jsonl")

In [49]:
write_jsonl(fine_tune_validation, "fine_tune_validation.jsonl")

In [50]:
with open("fine_tune_train.jsonl", "rb") as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")

In [51]:
train_file

FileObject(id='file-B5wEok5CZyy4k5VfAHCWuR', bytes=188742, created_at=1755512227, filename='fine_tune_train.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

In [52]:
with open("fine_tune_validation.jsonl", "rb") as f:
    validation_file = openai.files.create(file=f, purpose="fine-tune")

In [53]:
validation_file

FileObject(id='file-DgqMrkSjQrdUZVWCdaa1VC', bytes=47085, created_at=1755512233, filename='fine_tune_validation.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

# Step 2

I love Weights and Biases - a beautiful, free platform for monitoring training runs.  
Weights and Biases is integrated with OpenAI for fine-tuning.

First set up your weights & biases free account at:

https://wandb.ai

From the Avatar >> Settings menu, near the bottom, you can create an API key.

Then visit the OpenAI dashboard at:

https://platform.openai.com/account/organization

In the integrations section, you can add your Weights & Biases key.

## And now time to Fine-tune!

In [54]:

wandb_integration = {
    "type": "wandb",
    "wandb": {
        "project": "gpt-pricer",              # ✅ Aynı proje adı
        "entity": "llm-code08-thinka"         # ✅ Organization adı — kişisel kullanıcı adı değil!
    }
}


In [55]:
train_file.id

'file-B5wEok5CZyy4k5VfAHCWuR'

In [56]:
job = openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=validation_file.id,
    model="gpt-4o-mini-2024-07-18",
    hyperparameters={"n_epochs": 1},
    suffix="pricer",
    integrations=[wandb_integration],     # ✅ Bu satır kritik
    seed=42
)


In [57]:
job_id = job.id
print(job_id)


ftjob-YjutiCy4tUjEpegtymOdcCPy


In [58]:
job = openai.fine_tuning.jobs.retrieve("ftjob-tYQs1OwF2U4ZEqMc7WKrH3E7")
job.integrations

[FineTuningJobWandbIntegrationObject(type='wandb', wandb=FineTuningJobWandbIntegration(project='gpt-pricer', entity='llm-code08-thinka', name=None, tags=None, run_id='ftjob-tYQs1OwF2U4ZEqMc7WKrH3E7'))]

In [59]:
job = openai.fine_tuning.jobs.retrieve("ftjob-tYQs1OwF2U4ZEqMc7WKrH3E7")
print(job.status)

succeeded


In [60]:
events = openai.fine_tuning.jobs.list_events(fine_tuning_job_id=job.id, limit=20).data
for e in events:
    if e.type == "metrics":
        step = e.data.get("step")
        t_loss = e.data.get("train_loss")
        v_loss = e.data.get("valid_loss")
        t_acc = e.data.get("train_mean_token_accuracy")
        v_acc = e.data.get("valid_mean_token_accuracy")
        print(f"Step {step:>3} | Train Loss: {t_loss:.2f} | Val Loss: {v_loss} | Train Acc: {t_acc} | Val Acc: {v_acc}")


Step 200 | Train Loss: 1.14 | Val Loss: 1.1211233139038086 | Train Acc: 0.75 | Val Acc: 0.75
Step 199 | Train Loss: 1.43 | Val Loss: None | Train Acc: 0.75 | Val Acc: None
Step 198 | Train Loss: 0.52 | Val Loss: None | Train Acc: 0.875 | Val Acc: None
Step 197 | Train Loss: 1.26 | Val Loss: None | Train Acc: 0.75 | Val Acc: None
Step 196 | Train Loss: 0.85 | Val Loss: None | Train Acc: 0.875 | Val Acc: None
Step 195 | Train Loss: 1.25 | Val Loss: None | Train Acc: 0.75 | Val Acc: None
Step 194 | Train Loss: 0.91 | Val Loss: None | Train Acc: 0.875 | Val Acc: None
Step 193 | Train Loss: 1.28 | Val Loss: None | Train Acc: 0.75 | Val Acc: None
Step 192 | Train Loss: 1.44 | Val Loss: None | Train Acc: 0.75 | Val Acc: None
Step 191 | Train Loss: 1.07 | Val Loss: None | Train Acc: 0.75 | Val Acc: None
Step 190 | Train Loss: 0.92 | Val Loss: 0.9432516098022461 | Train Acc: 0.75 | Val Acc: 0.75
Step 189 | Train Loss: 1.79 | Val Loss: None | Train Acc: 0.75 | Val Acc: None
Step 188 | Train Loss

In [61]:
print(job.integrations)


[FineTuningJobWandbIntegrationObject(type='wandb', wandb=FineTuningJobWandbIntegration(project='gpt-pricer', entity='llm-code08-thinka', name=None, tags=None, run_id='ftjob-tYQs1OwF2U4ZEqMc7WKrH3E7'))]


In [62]:
events = openai.fine_tuning.jobs.list_events(fine_tuning_job_id="ftjob-tYQs1OwF2U4ZEqMc7WKrH3E7", limit=50)
for event in events.data[::-1]:  # en güncel en altta olsun
    print(f"{event.created_at} | {event.message}")


1755356857 | Step 156/200: training loss=0.56
1755356860 | Step 157/200: training loss=1.74
1755356860 | Step 158/200: training loss=1.65
1755356860 | Step 159/200: training loss=1.26
1755356865 | Step 160/200: training loss=1.49, validation loss=1.58
1755356865 | Step 161/200: training loss=0.78
1755356868 | Step 162/200: training loss=0.87
1755356868 | Step 163/200: training loss=1.49
1755356868 | Step 164/200: training loss=1.23
1755356871 | Step 165/200: training loss=1.28
1755356871 | Step 166/200: training loss=1.30
1755356871 | Step 167/200: training loss=0.56
1755356874 | Step 168/200: training loss=1.36
1755356874 | Step 169/200: training loss=1.02
1755356877 | Step 170/200: training loss=1.27, validation loss=1.26
1755356880 | Step 171/200: training loss=1.35
1755356880 | Step 172/200: training loss=1.48
1755356880 | Step 173/200: training loss=0.77
1755356882 | Step 174/200: training loss=0.86
1755356882 | Step 175/200: training loss=0.90
1755356882 | Step 176/200: training 

In [63]:
job = openai.fine_tuning.jobs.retrieve("ftjob-tYQs1OwF2U4ZEqMc7WKrH3E7")
print(job.fine_tuned_model)


ft:gpt-4o-mini-2024-07-18:gpt-pricer:pricer:C5Ct3RHj


In [64]:
openai.fine_tuning.jobs.list(limit=1)

SyncCursorPage[FineTuningJob](data=[FineTuningJob(id='ftjob-YjutiCy4tUjEpegtymOdcCPy', created_at=1755512257, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs=1), model='gpt-4o-mini-2024-07-18', object='fine_tuning.job', organization_id='org-SNRWYLg1yJl3jjiOiwbIh3xX', result_files=[], seed=42, status='validating_files', trained_tokens=None, training_file='file-B5wEok5CZyy4k5VfAHCWuR', validation_file='file-DgqMrkSjQrdUZVWCdaa1VC', estimated_finish=None, integrations=[FineTuningJobWandbIntegrationObject(type='wandb', wandb=FineTuningJobWandbIntegration(project='gpt-pricer', entity='llm-code08-thinka', name=None, tags=None, run_id='ftjob-YjutiCy4tUjEpegtymOdcCPy'))], metadata=None, method=Method(dpo=None, supervised=MethodSupervised(hyperparameters=MethodSupervisedHyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs=1)), type='supe

In [65]:
job_id = openai.fine_tuning.jobs.list(limit=1).data[0].id

In [66]:
job_id

'ftjob-YjutiCy4tUjEpegtymOdcCPy'

In [67]:
openai.fine_tuning.jobs.retrieve(job_id)

FineTuningJob(id='ftjob-YjutiCy4tUjEpegtymOdcCPy', created_at=1755512257, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs=1), model='gpt-4o-mini-2024-07-18', object='fine_tuning.job', organization_id='org-SNRWYLg1yJl3jjiOiwbIh3xX', result_files=[], seed=42, status='validating_files', trained_tokens=None, training_file='file-B5wEok5CZyy4k5VfAHCWuR', validation_file='file-DgqMrkSjQrdUZVWCdaa1VC', estimated_finish=None, integrations=[FineTuningJobWandbIntegrationObject(type='wandb', wandb=FineTuningJobWandbIntegration(project='gpt-pricer', entity='llm-code08-thinka', name=None, tags=None, run_id='ftjob-YjutiCy4tUjEpegtymOdcCPy'))], metadata=None, method=Method(dpo=None, supervised=MethodSupervised(hyperparameters=MethodSupervisedHyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs=1)), type='supervised'), user_provided_suffix='pric

In [68]:
openai.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10).data

[FineTuningJobEvent(id='ftevent-TfrxFs0OMjLGhp24o7CgrijQ', created_at=1755512257, level='info', message='Validating training file: file-B5wEok5CZyy4k5VfAHCWuR and validation file: file-DgqMrkSjQrdUZVWCdaa1VC', object='fine_tuning.job.event', data={}, type='message'),
 FineTuningJobEvent(id='ftevent-IAcbGDboWPqAbbQYxYSx1oab', created_at=1755512257, level='info', message='Created fine-tuning job: ftjob-YjutiCy4tUjEpegtymOdcCPy', object='fine_tuning.job.event', data={}, type='message')]

# Step 3

Test our fine tuned model

In [69]:
fine_tuned_model_name = openai.fine_tuning.jobs.retrieve(job_id).fine_tuned_model

In [70]:
fine_tuned_model_name

In [71]:
# The prompt

def messages_for(item):
    system_message = "You estimate prices of items. Reply only with the price, no explanation"
    user_prompt = item.test_prompt().replace(" to the nearest dollar","").replace("\n\nPrice is $","")
    return [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_prompt},
        {"role": "assistant", "content": "Price is $"}
    ]

In [72]:
# Try this out

messages_for(test[0])

[{'role': 'system',
  'content': 'You estimate prices of items. Reply only with the price, no explanation'},
 {'role': 'user',
  'content': "How much does this cost?\n\nOEM AC Compressor w/A/C Repair Kit For Ford F150 F-150 V8 & Lincoln Mark LT 2007 2008 - BuyAutoParts NEW\nAs one of the world's largest automotive parts suppliers, our parts are trusted every day by mechanics and vehicle owners worldwide. This A/C Compressor and Components Kit is manufactured and tested to the strictest OE standards for unparalleled performance. Built for trouble-free ownership and 100% visually inspected and quality tested, this A/C Compressor and Components Kit is backed by our 100% satisfaction guarantee. Guaranteed Exact Fit for easy installation 100% BRAND NEW, premium ISO/TS 16949 quality - tested to meet or exceed OEM specifications Engineered for superior durability, backed by industry-leading unlimited-mileage warranty Included in this K"},
 {'role': 'assistant', 'content': 'Price is $'}]

In [73]:
# A utility function to extract the price from a string

def get_price(s):
    s = s.replace('$','').replace(',','')
    match = re.search(r"[-+]?\d*\.\d+|\d+", s)
    return float(match.group()) if match else 0

In [74]:
get_price("The price is roughly $99.99 because blah blah")

99.99

In [75]:
# The function for gpt-4o-mini

def gpt_fine_tuned(item):
    response = openai.chat.completions.create(
        model=fine_tuned_model_name, 
        messages=messages_for(item),
        seed=42,
        max_tokens=7
    )
    reply = response.choices[0].message.content
    return get_price(reply)

In [76]:
print(test[0].price)
print(gpt_fine_tuned(test[0]))

374.41


BadRequestError: Error code: 400 - {'error': {'message': 'you must provide a model parameter', 'type': 'invalid_request_error', 'param': None, 'code': None}}

In [77]:
print(test[0].test_prompt())

How much does this cost to the nearest dollar?

OEM AC Compressor w/A/C Repair Kit For Ford F150 F-150 V8 & Lincoln Mark LT 2007 2008 - BuyAutoParts NEW
As one of the world's largest automotive parts suppliers, our parts are trusted every day by mechanics and vehicle owners worldwide. This A/C Compressor and Components Kit is manufactured and tested to the strictest OE standards for unparalleled performance. Built for trouble-free ownership and 100% visually inspected and quality tested, this A/C Compressor and Components Kit is backed by our 100% satisfaction guarantee. Guaranteed Exact Fit for easy installation 100% BRAND NEW, premium ISO/TS 16949 quality - tested to meet or exceed OEM specifications Engineered for superior durability, backed by industry-leading unlimited-mileage warranty Included in this K

Price is $


In [78]:
Tester.test(gpt_fine_tuned, test)

BadRequestError: Error code: 400 - {'error': {'message': 'you must provide a model parameter', 'type': 'invalid_request_error', 'param': None, 'code': None}}

In [79]:
import random
import wandb

run = wandb.init(
    entity="llm-code08-thinka",          # ✅ kendi organizasyon adın
    project="gpt-pricer",                # ✅ fine-tuning sırasında kullandığın proje adı
    config={
        "learning_rate": 0.01,
        "architecture": "TestNet",
        "dataset": "DummySet",
        "epochs": 10,
    },
)

# Dummy eğitim simülasyonu
epochs = 10
for epoch in range(epochs):
    acc = 0.8 + random.uniform(-0.05, 0.05)
    loss = 0.6 + random.uniform(-0.05, 0.05)
    run.log({"epoch": epoch, "accuracy": acc, "loss": loss})

run.finish()


accuracy,▅▆▄▄▇▁▃▂▇█
epoch,▁▂▃▃▄▅▆▆▇█
loss,▅█▂█▂▁▁▇▄▄
accuracy,0.84642
epoch,9
loss,0.58951
